# WAN stats report

Loss and latency for every Meraki uplink in the database, read as a data set: checks on the data, the links whose latency changed, every dated level shift, and every outage.

In [ ]:
from IPython.display import Markdown, display

from wanstats import analysis, charts, config, notebook

DB = None         # None reads WAN_STATS_DB from .env. For the invented demo data: 'sqlite:///demo.sqlite'
DAYS = 7          # the period of the summary table and the site charts
THEME = 'light'   # or 'dark'

store = config.open_db(DB, create=False)
a = analysis.analyse(store.load())
store.close()
print('%d links analysed, %d flagged' % (len(a.links), len(a.flagged)))

## 1. Data checks

Before any statistics, the data itself: how much there is, how complete it is, and how much of it is unusable. A stretch where most of an organization's links went quiet at the same moment is the collector failing, not the sites, so it is kept out of every outage count.

In [ ]:
notebook.checks(a)

## 2. Every link, every week

One row per link and one cell per week. A cell is shaded by how far that week's median latency sits above the link's own baseline, which is the lowest level the link has held for 30 days or more. Links are never compared with each other.

In [ ]:
charts.fleet(a, THEME)

## 3. Flagged links

A link is **Worse** when its latency sits at least 50% and 10 ms above its baseline, and **Drifting** at 10% and 3 ms. The rise is found in one of two ways: a level shift with a date, or a trend inside the current level (Theil-Sen slope, Mann-Kendall p below 0.001). The median and the 95th percentile are each tested, and a link takes the worse verdict.

In [ ]:
display(notebook.flagged(a))
for link in a.flagged:
    display(charts.flagged(a, link, THEME))

## 4. Level shifts

Every day, the 14 days after are compared with the 14 days before. A shift is a change in the median of at least 3 ms, 10%, and four times the link's day-to-day noise. Improvements are found as readily as slowdowns. Shifts on several links within a few days of each other usually share a cause outside the sites, such as an ISP or the path to the test address.

In [ ]:
display(charts.shifts(a, THEME))
notebook.shifts(a)

## 5. The last week, link by link

The same table as the console report. **Median** is the typical minute and **p95** the slow end: one minute in twenty is slower. **Bad min** is the share of minutes with 2% loss or more, or latency over 150 ms. **Avail** is the share of the period the link was not in an outage.

In [ ]:
notebook.summary(a, DAYS)

## 6. Outages

An outage is three minutes or more of at least 80% loss, or more than ten minutes with no readings from a link while the rest of its organization kept reporting.

In [ ]:
display(charts.downtime(a, THEME))
notebook.outages(a, top=6)

## 7. Sites with two uplinks

Where a site has two uplinks and one of them had an outage in the period, both are drawn together.

In [ ]:
sites = notebook.dual_sites(a, DAYS)
for org, name in sites:
    display(charts.site(a, name, DAYS, THEME, org=org))
if not sites:
    print('No site with two uplinks had an outage in the last %d days.' % DAYS)

## 8. Latency distribution

One box per link for everything stored. A wide box or a long line means the link spent part of the time at a different level, or that its latency varies a lot from minute to minute.

In [ ]:
charts.boxes(a, THEME)

## 9. Your own analysis

Everything above comes from `a`. Each link in `a.links` carries its readings, its daily values, its verdict and its outages. `notebook.frame()` gives one link's readings as a pandas DataFrame.

In [ ]:
link = a.flagged[0] if a.flagged else a.links[0]
frame = notebook.frame(link)
print(link.label)
frame.set_index('time')['latency_ms'].resample('30D').agg(['count', 'median', 'max']).round(1)

## 10. Exporting this report

Run the cells from top to bottom, or let nbconvert run them and write the report with the code hidden:

```
jupyter nbconvert --to webpdf --no-input --execute wan-stats.ipynb
jupyter nbconvert --to html --no-input --execute wan-stats.ipynb
```